In [ ]:
from statistics import correlation, linear_regression

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

In [ ]:
data = pd.read_csv('insurance.csv')
# data.info()
data.describe()


In [ ]:
data.isnull().sum()


In [ ]:
data.columns

In [ ]:
numeric_colums=['age','bmi','children','charges']
for col in numeric_colums:
    plt.figure(figsize=(6,4))
    sns.histplot(data[col],kde = True,bins= 20)

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(data.corr(numeric_only=True),annot=True)

#data cleanning


In [ ]:
df_cleaned = data.copy()
df_cleaned.head()


In [ ]:
df_cleaned.drop_duplicates(inplace=True)
df_cleaned.shape

In [ ]:
df_cleaned["sex"].value_counts()

In [ ]:
#labeling male as 0 female as 1
df_cleaned['sex'] = df_cleaned['sex'].map({"male" : 0, "female" : 1} )
df_cleaned.head()

In [ ]:
df_cleaned['smoker'].value_counts()

In [ ]:
df_cleaned['smoker'] = df_cleaned['smoker'].map({'no' : 0 , 'yes' : 1})
df_cleaned.head()

In [ ]:
df_cleaned.rename(columns={
    'sex' : 'is_female',
    'smoker' : 'is smoker',
                            },inplace=True)

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned = pd.get_dummies(df_cleaned,columns= ['region'],drop_first=True)
df_cleaned.head()

In [ ]:
df_cleaned = df_cleaned.astype(int)
df_cleaned.head()

##Feature ENg

In [ ]:
#adding extra colums with bmi category

In [ ]:
df_cleaned['bmi_category'] = pd.cut(df_cleaned['bmi'],bins=[0,18.5,24.9,29.9,float('inf')],labels=['underweight','normal', 'overweight', 'obese'])

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned = pd.get_dummies(df_cleaned, columns=['bmi_category'], drop_first= True)
df_cleaned = df_cleaned.astype(int)
df_cleaned.head()

In [ ]:
from sklearn.preprocessing import StandardScaler
cols = ['age','bmi','children']
scaler = StandardScaler()
df_cleaned[cols] = scaler.fit_transform(df_cleaned[cols])

In [ ]:
df_cleaned.head()


In [ ]:
from scipy.stats import pearsonr


selected_features = [
    'age', 'bmi','children','is_female','is smoker',
    'region_northwest','region_southwest','region_southeast','bmi_category_normal',
    'bmi_category_overweight','bmi_category_obese'
]

correlation = {
    feature : pearsonr(df_cleaned[feature], df_cleaned['charges'])[0]
    for feature in selected_features
}

correlation_df = pd.DataFrame(list(correlation.items()),columns=['feature', 'pearson Correlation'])
correlation_df.sort_values(by='pearson Correlation',ascending= False)

In [ ]:
x = df_cleaned.drop(['is smoker','age', 'bmi'],axis=1)
y = df_cleaned['charges']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train,y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("R² Score:", r2)